# Importação de Bibliotecas e Configuração da Conexão

In [1]:
import pandas as pd
import datetime
import logging
import os
import numpy as np
from sqlalchemy import create_engine, text, inspect, types
from urllib.parse import quote_plus
from dotenv import load_dotenv

In [2]:
# Carrega variáveis de ambiente
load_dotenv()

print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


In [3]:
# Definindo a função para criar a engine do banco de dados
def create_db_engine():
        user = quote_plus(os.getenv('DB_USER', 'postgres'))
        password = quote_plus(os.getenv('DB_PASS', 'password'))
        host = os.getenv('DB_HOST', 'localhost')
        port = os.getenv('DB_PORT', '5432')
        dbname = os.getenv('DB_NAME', 'data_lake')
        
        connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        return create_engine(connection_str)

In [4]:
# Efetivamente criando a engine
engine = create_db_engine()

# Carga e Extração das Variáveis Nível 1 (Mapeamento do Data Contract)
Nesta etapa, extraímos as contas específicas de cada demonstrativo financeiro da camada Silver utilizando os filtros exatos de CD_CONTA definidos no contrato de dados.

In [5]:
print("Carregando e filtrando dados da camada Silver...")

# 1. Variáveis oriundas do Balanço Patrimonial (BP)
query_bp = """
SELECT "CNPJ_CIA", "DT_REFER", "DENOM_CIA" AS "RAZAO_SOCIAL", "SETOR_ATIV" AS "SETOR", "CD_CONTA", "VL_CONTA_TRATADO"
FROM layer_02_silver.n1_dfp_cia_aberta_bp
WHERE "CD_CONTA" IN ('1.01', '1.02', '1.02.01', '2.01', '2.02', '1.01.04', '1.01.01', '1.01.02', '2.03', '1.02.03', '1.01.03', '2.01.02', '2.01.04', '2.02.01')
"""
df_bp_raw = pd.read_sql(query_bp, con=engine)

# Mapeamento do Data Contract para o BP
map_bp = {
    '1.01': 'V1_AC', '1.02': 'V1_ANC', '1.02.01': 'V1_RLP', '2.01': 'V1_PC', 
    '2.02': 'V1_PNC', '1.01.04': 'V1_ES', '1.01.01': 'V1_CX', '1.01.02': 'V1_AP', 
    '2.03': 'V1_PL', '1.02.03': 'V1_AI', '1.01.03': 'V1_CR', '2.01.02': 'V1_FO',
    '2.01.04': 'V1_EFCP', '2.02.01': 'V1_EFLP'
}
df_bp_raw['VARIAVEL'] = df_bp_raw['CD_CONTA'].map(map_bp)

# Nota: Como o código '2.01.02' mapeia tanto para V1_CP quanto V1_FO no contrato, duplicamos essa informação
df_fo_to_cp = df_bp_raw[df_bp_raw['VARIAVEL'] == 'V1_FO'].copy()
df_fo_to_cp['VARIAVEL'] = 'V1_CP'
df_bp_raw = pd.concat([df_bp_raw, df_fo_to_cp], ignore_index=True)


# 2. Variáveis oriundas do Demonstrativo de Resultados (DRE)
query_dre = """
SELECT "CNPJ_CIA", "DT_REFER_TRATADO" AS "DT_REFER", "DENOM_CIA" AS "RAZAO_SOCIAL", "SETOR_ATIV" AS "SETOR", "CD_CONTA", "VL_CONTA_TRATADO"
FROM layer_02_silver.n1_dfp_cia_aberta_dre
WHERE "CD_CONTA" IN ('3.03', '3.01', '3.11', '3.05', '3.02')
"""
df_dre_raw = pd.read_sql(query_dre, con=engine)

# Mapeamento do Data Contract para a DRE
map_dre = {
    '3.03': 'V1_LB', '3.01': 'V1_RL', '3.11': 'V1_LL', '3.05': 'V1_EBIT', '3.02': 'V1_COGS'
}
df_dre_raw['VARIAVEL'] = df_dre_raw['CD_CONTA'].map(map_dre)


# Consolidação das fontes brutas de movimentações contábeis
df_all_raw = pd.concat([df_bp_raw, df_dre_raw], ignore_index=True)

print("Dados contábeis carregados com sucesso!")

Carregando e filtrando dados da camada Silver...
Dados contábeis carregados com sucesso!


# Pivotagem dos Dados para o Formato Cadastral (Colunar)
Transformamos os dados que estão em formato "longo" (linhas com chaves e valores contábeis) para o formato "largo", onde cada variável de Nível 1 se torna uma coluna da nossa futura tabela Gold.

In [6]:
print("Buscando dados cadastrais e pivotando tabelas...")

# Carga do cadastro de empresas isolando o par único de CNPJ e TP_MERC para evitar junções duplicadas
query_cadastro = """
SELECT "CNPJ_CIA", "TP_MERC"
FROM layer_02_silver.n0_empresas_selecionadas
WHERE "CNPJ_CIA" IS NOT NULL
"""
df_cadastro = pd.read_sql(query_cadastro, con=engine)
df_cadastro = df_cadastro.drop_duplicates(subset=['CNPJ_CIA'])

# Pivotagem para transformar registros verticais em colunas horizontais de métricas
df_pivot = df_all_raw.pivot_table(
    index=['CNPJ_CIA', 'DT_REFER', 'RAZAO_SOCIAL', 'SETOR'],
    columns='VARIAVEL',
    values='VL_CONTA_TRATADO',
    aggfunc='first'
).reset_index()

# Cruzamento dos dados contábeis pivotados com o cadastro de empresas (Mapeamento de TP_MERC)
df_gold = pd.merge(df_pivot, df_cadastro, on='CNPJ_CIA', how='left')

# Garantir o preenchimento de colunas faltantes ou nulas com zero
v1_cols = [
    'V1_AC', 'V1_ANC', 'V1_RLP', 'V1_PC', 'V1_PNC', 'V1_ES', 'V1_CX', 'V1_AP', 
    'V1_PL', 'V1_AI', 'V1_LB', 'V1_RL', 'V1_CR', 'V1_CP', 'V1_FO', 'V1_LL', 
    'V1_EFCP', 'V1_EFLP', 'V1_EBIT', 'V1_COGS'
]
for col in v1_cols:
    if col not in df_gold.columns:
        df_gold[col] = 0.0
    else:
        df_gold[col] = df_gold[col].fillna(0.0)

# Tratamento do campo TP_MERC para não permitir valores nulos na tabela Gold
df_gold['TP_MERC'] = df_gold['TP_MERC'].fillna('NÃO INFORMADO')

# Ajuste do formato da data de referência
df_gold['DT_REFER'] = pd.to_datetime(df_gold['DT_REFER']).dt.date

print(f"Total de registros prontos para o cálculo: {df_gold.shape[0]}")

Buscando dados cadastrais e pivotando tabelas...
Total de registros prontos para o cálculo: 2221


# Processamento das Variáveis Nível 2 e Indicadores Financeiros
Nesta célula aplicamos rigorosamente as fórmulas matemáticas explícitas definidas no Contrato de Dados. Utilizamos o parâmetro .replace([np.inf, -np.inf], np.nan) e tratamento de divisões por zero para evitar que erros matemáticos quebrem a execução do script.

In [7]:
print("Calculando variáveis de Nível 2 e Indicadores Financeiros...")

# --- Variáveis Nível 2 ---
df_gold['V2_AT'] = df_gold['V1_AC'] + df_gold['V1_ANC']
df_gold['V2_EFT'] = df_gold['V1_EFCP'] + df_gold['V1_EFLP']

# --- Indicadores: Liquidez ---
df_gold['I_LIQ_GER'] = (df_gold['V1_AC'] + df_gold['V1_RLP']) / (df_gold['V1_PC'] + df_gold['V1_PNC'])
df_gold['I_LIQ_COR'] = df_gold['V1_AC'] / df_gold['V1_PC']
df_gold['I_LIQ_SEC'] = (df_gold['V1_AC'] - df_gold['V1_ES']) / df_gold['V1_PC']
df_gold['I_LIQ_IME'] = (df_gold['V1_CX'] + df_gold['V1_AP']) / df_gold['V1_PC']

# --- Indicadores: Endividamento ---
df_gold['I_PCT_CP'] = (df_gold['V1_PC'] + df_gold['V1_PNC']) / df_gold['V1_PL']
df_gold['I_PCT_AT'] = (df_gold['V1_PC'] + df_gold['V1_PNC']) / df_gold['V2_AT']
df_gold['I_GAR_CP_CT'] = df_gold['V1_PL'] / (df_gold['V1_PC'] + df_gold['V1_PNC'])
df_gold['I_COMP_ENDIV'] = df_gold['V1_PC'] / (df_gold['V1_PC'] + df_gold['V1_PNC'])
df_gold['I_IMOB_PL'] = df_gold['V1_AI'] / df_gold['V1_PL']
df_gold['I_IMOB_AT'] = df_gold['V1_AI'] / df_gold['V2_AT']

# --- Indicadores: Margem de Lucro ---
df_gold['I_MARGEM_LUCR'] = df_gold['V1_LB'] / df_gold['V1_RL']
df_gold['I_MARGEM_OPER'] = df_gold['V1_EBIT'] / df_gold['V1_RL']
df_gold['I_MARGEM_LIQ'] = df_gold['V1_LL'] / df_gold['V1_RL']

# --- Indicadores: Rentabilidade ---
df_gold['I_ROA'] = df_gold['V1_LL'] / df_gold['V2_AT']
df_gold['I_ROE'] = df_gold['V1_LL'] / df_gold['V1_PL']
df_gold['I_ROI'] = df_gold['V1_LL'] / (df_gold['V2_EFT'] + df_gold['V1_PL'])

# --- Indicadores: Atividade ---
df_gold['I_GIR_ESTQ'] = df_gold['V1_COGS'] / df_gold['V1_ES']
df_gold['I_GIR_CR'] = df_gold['V1_RL'] / df_gold['V1_CR']
df_gold['I_GIR_CP'] = df_gold['V1_COGS'] / df_gold['V1_FO']
df_gold['I_GIR_AC'] = df_gold['V1_RL'] / df_gold['V1_AC']
df_gold['I_PMRE'] = (df_gold['V1_ES'] * 360) / df_gold['V1_COGS']
df_gold['I_PMRV'] = (df_gold['V1_CR'] * 360) / df_gold['V1_RL']
df_gold['I_PMPC'] = (df_gold['V1_FO'] * 360) / df_gold['V1_COGS']
df_gold['I_PMRAC'] = (df_gold['V1_AC'] * 360) / df_gold['V1_RL']

# --- Indicadores: Ciclos ---
df_gold['I_CICLO_ECON'] = df_gold['I_PMRE'] + df_gold['I_PMRV']
df_gold['I_CICLO_FIN'] = df_gold['I_PMRE'] + df_gold['I_PMRV'] - df_gold['I_PMPC']

# --- Indicadores: Recursos Financeiros ---
df_gold['I_CGL_CCL'] = df_gold['V1_AC'] - df_gold['V1_PC']
df_gold['I_NCG'] = (df_gold['V1_ES'] + df_gold['V1_CR']) - df_gold['V1_FO']
df_gold['I_ST'] = df_gold['V1_AC'] - df_gold['V1_EFCP']

# Substituição de infinitos matemáticos decorrentes de divisões por zero
df_gold = df_gold.replace([np.inf, -np.inf], np.nan)

print("Cálculos dos indicadores finalizados!")

Calculando variáveis de Nível 2 e Indicadores Financeiros...
Cálculos dos indicadores finalizados!


# Ordenação e Carga Final na Camada Gold (mart_indicadores_financeiros)
Aqui garantimos que a estrutura de colunas obedeça estritamente à ordem mapeada no seu arquivo DDL Gold, realizando uma inserção em lote eficiente.

In [8]:
print("Preparando dados estruturados para inserção...")

# Sequenciamento exato conforme a assinatura DDL da tabela Gold
colunas_destino = [
    "CNPJ_CIA", "DT_REFER", "RAZAO_SOCIAL", "SETOR", "TP_MERC",
    "V1_AC", "V1_ANC", "V1_RLP", "V1_PC", "V1_PNC", "V1_ES", "V1_CX", "V1_AP", "V1_PL", "V1_AI", 
    "V1_LB", "V1_RL", "V1_CR", "V1_CP", "V1_FO", "V1_LL", "V1_EFCP", "V1_EFLP", "V1_EBIT", "V1_COGS",
    "V2_AT", "V2_EFT",
    "I_LIQ_GER", "I_LIQ_COR", "I_LIQ_SEC", "I_LIQ_IME",
    "I_PCT_CP", "I_PCT_AT", "I_GAR_CP_CT", "I_COMP_ENDIV", "I_IMOB_PL", "I_IMOB_AT",
    "I_MARGEM_LUCR", "I_MARGEM_OPER", "I_MARGEM_LIQ",
    "I_ROA", "I_ROE", "I_ROI",
    "I_GIR_ESTQ", "I_GIR_CR", "I_GIR_CP", "I_GIR_AC", "I_PMRE", "I_PMRV", "I_PMPC", "I_PMRAC",
    "I_CICLO_ECON", "I_CICLO_FIN",
    "I_CGL_CCL", "I_NCG", "I_ST"
]

# Reindexação estrita para conformidade posicional
df_carga_gold = df_gold.reindex(columns=colunas_destino)

# Persistência final via SQLAlchemy no schema Gold
df_carga_gold.to_sql(
    name='mart_indicadores_financeiros_x',
    schema='layer_03_gold',
    con=engine,
    if_exists='append',
    index=False,
    chunksize=1000,
    dtype={
        'CNPJ_CIA': types.VARCHAR(14),
        'DT_REFER': types.Date,
        'RAZAO_SOCIAL': types.VARCHAR(255),
        'SETOR': types.VARCHAR(100),
        'TP_MERC': types.VARCHAR(50)
    }
)

print(f"Pipeline concluída! {len(df_carga_gold)} registros indexados com o tipo de mercado correspondente inseridos na Gold.")

Preparando dados estruturados para inserção...
Pipeline concluída! 2221 registros indexados com o tipo de mercado correspondente inseridos na Gold.
